# Minimal IsoCLIP Demo

This notebook demonstrates how to:
1. Load a CLIP model
2. Extract pre-projection features
3. Compute IsoCLIP projectors
4. Apply them in a forward pass


In [1]:
import sys
from pathlib import Path
project_root = Path.cwd()  # IsoCLIP/
sys.path.append(str(project_root / "src"))

import torch
import open_clip.transformer
from functools import partial
import torch.nn.functional as F
from utils import load_clip
from retrieval import apply_iso
from encode_no_projection import (
    get_projection_layers,
    get_encode_image_with_noproj,
    get_encode_text_with_noproj,
    encode_attention_module,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

/home/smagistri/miniconda3/envs/iso-clip/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
# -------------------------
# Model configuration
# -------------------------
clip_model_name = "ViT-B/32"
use_open_clip = False # True
open_clip_pretrained = None   # "datacomp_xl_s13b_b90k"

# -------------------------
# IsoCLIP hyperparameters
# -------------------------
# Number of top / bottom spectral directions removed
iso_ktop = 150
iso_kbottom = 50

In [3]:
# -------------------------
# Load CLIP model
# -------------------------
clip_model, clip_model_name, preprocess = load_clip(
    clip_model_name,
    open_clip_pretrained,
    use_open_clip,
    device,
)
clip_model.eval()

print("Loaded:", clip_model_name)

Loading OpenAI CLIP model:  ViT-B/32
Loaded: ViT-B/32


In [4]:
# -------------------------
# Modify forward pass
# -------------------------
# We override the default CLIP forward to extract
# **pre-projection image features**, i.e., features
# before the final projection layer W_image.
#
# This is required because IsoCLIP operates directly
# on the projection matrices.

if isinstance(clip_model.visual, open_clip.timm_model.TimmModel):
    # For Perception-encoder and Siglip-models
    clip_model.visual.trunk.attn_pool.forward = partial(
        encode_attention_module,
        clip_model.visual.trunk.attn_pool,
    )
    encode_image_noproj = get_encode_image_with_noproj(clip_model)
    clip_model.encode_image = partial(encode_image_noproj, clip_model)
else:
    encode_image_noproj = get_encode_image_with_noproj(clip_model)
    clip_model.encode_image = partial(encode_image_noproj, clip_model)

print("Forward modified for pre-projection features")

Forward modified for pre-projection features


In [5]:
# -------------------------
# Extract projection layers
# -------------------------
# W_image and W_text map features into the shared CLIP space 

W_image, W_text = get_projection_layers(clip_model, clip_model_name)
W_image = W_image.T
W_text = W_text.T

print("W_image:", W_image.shape)
print("W_text :", W_text.shape)

W_image: torch.Size([512, 768])
W_text : torch.Size([512, 512])


In [6]:
# -------------------------
# Apply IsoCLIP
# -------------------------
 
W_text_iso, W_image_iso = apply_iso(
    W_text,
    W_image,
    iso_ktop=iso_ktop,
    iso_kbottom=iso_kbottom,
)

print("W_image_iso:", W_image_iso.shape)
print("W_text_iso :", W_text_iso.shape)

Manual filtering: k_top = 150, k_bottom = 50
W_image_iso: torch.Size([768, 512])
W_text_iso : torch.Size([512, 512])


In [7]:
# -------------------------
# Simulated forward pass
# -------------------------
# Example: apply IsoCLIP projection to image features 

image_size = 224
x = torch.randn(4, 3, image_size, image_size, device=device)
with torch.no_grad():
    f_img_pre = clip_model.encode_image(x)
    if clip_model_name == "ViT-B-16-SigLIP2-webli":
        ones_q =  f_img_pre.new_ones(f_img_pre.size(0), 1)    
        f_img_pre = torch.cat([f_img_pre, ones_q], dim=1)      
    
    # Apply IsoCLIP projector
    f_img_iso = f_img_pre @ W_image_iso
    f_img_iso = F.normalize(f_img_iso, dim=-1)
    print(f_img_iso.shape)

torch.Size([4, 512])


In [8]:
# -------------------------
# Simulated forward pass
# -------------------------
# Example: apply IsoCLIP projection to text features  

encode_text_noproj = get_encode_text_with_noproj(clip_model)
clip_model.encode_text = partial(encode_text_noproj, clip_model)

# -------------------------
# Simulate text input
# -------------------------
texts = [
    "a dog",
    "a cat",
    "a car",
    "a random object"
]

text_tokens = clip_model.tokenizer(texts).to(device)
with torch.no_grad():
    f_txt_pre = clip_model.encode_text(text_tokens)
    
# -------------------------
# Apply IsoCLIP
# -------------------------
f_txt_iso = f_txt_pre @ W_text_iso
f_txt_iso = torch.nn.functional.normalize(f_txt_iso, dim=-1)
    